# Phase 4 — LLM Fine-tuning (V10 enriched YoY + corridor ranking)

**Branch:** `v10_enriched_yoy_corridors`

This notebook is the **V9 forecast-only LoRA run from scratch**. There is **no warm-start from V4** here: the goal is to test whether a cleaner task definition and a stronger traffic-memory prompt can improve forecast behavior on their own.

**V9 training strategy:**
1. Start from the base instruct model with `init_adapter_path = None`
2. Train on the V9 forecast dataset produced by `04_textualize.ipynb`
3. Prioritize **structured forecast outputs** over long prose
4. Select the final adapter on **validation forecast MAE**, not on eval loss alone

**Forecast output contract in V9:**
```json
{"next_month_am_tt_ratio": 1.250, "delta_vs_current": -0.010}
```

**Why this matters:** previous branches often collapsed toward a handful of memorized prototype values. V9 tries to reduce that collapse by tightening the task, the prompt, and the extraction path around the numeric forecast itself.


In [1]:
# ── 0. GPU CHECK ──────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('WARNING: nvidia-smi not found.')
    print('This notebook requires a CUDA GPU (RTX 4070 12GB / 4090 24GB / 5090 32GB).')

import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f'\nGPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {vram:.1f} GB')
    print(f'CUDA     : {torch.version.cuda}')
    print(f'PyTorch  : {torch.__version__}')
    if vram < 10:
        print('\n⚠️  VRAM < 10GB — may struggle with 8B model even in 4-bit.')
        print('   Consider: microsoft/Phi-3-mini-4k-instruct (3.8B, ~3GB VRAM)')
    elif vram < 16:
        print('\n✓ 4-bit QLoRA recommended for this VRAM (load_in_4bit=True in CONFIG)')
    else:
        print('\n✓ 8-bit or FP16 feasible — can replicate paper settings exactly')
else:
    print('\nERROR: CUDA not available. Run this notebook on a GPU machine via SSH.')

Tue Apr 28 14:24:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   38C    P8              7W /  320W |     659MiB /  16376MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, getpass

if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")


In [3]:
# -- 1. CONFIG - MODIFIER ICI SELON LA MACHINE -------------------------------
import os

CONFIG = {
    # -- Experiment -----------------------------------------------------------
    'experiment_name': 'v10_enriched_yoy_corridors',

    # -- Modèle ---------------------------------------------------------------
    'model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct',

    'hf_token': os.getenv('HF_TOKEN', None),
    'openai_api_key': os.getenv('OPENAI_API_KEY', None),

    'load_in_4bit': True,
    'load_in_8bit': False,

    'lora_r':           16,
    'lora_alpha':       32,
    'lora_dropout':     0.10,
    'target_modules':   ['q_proj', 'v_proj', 'k_proj', 'o_proj'],

    'num_epochs':                   3,
    'per_device_train_batch_size':  4,
    'gradient_accumulation_steps':  2,
    'learning_rate':                2e-4,
    'warmup_ratio':                 0.05,
    'max_seq_length':               3072,
    'fp16':                         True,
    'temperature':                  0.95,
    'max_new_tokens':               256,
    'init_adapter_path':            None,

    'ablation_variant': 'full',

    'data_path':    '../data/llm_prompts_v10_enriched_yoy_corridors.jsonl',
    'dataset_csv':  '../data/ml_dataset_gpu.csv',
    'val_csv':      '../data/ml_val.csv',
    'test_csv':     '../data/ml_test.csv',
    'output_dir':   '../llm/outputs/lora_v10_enriched_yoy_corridors',
    'results_path': '../data/llm_results_v10_enriched_yoy_corridors.csv',
    'expl_path':    '../data/explainability_scores_v10_enriched_yoy_corridors.csv',
    'zs_path':      '../llm/outputs/zero_shot_results_v10_enriched_yoy_corridors.csv',
    'predictions_path': '../data/llm_predictions_test_forecast_h1_v10_enriched_yoy_corridors.csv',
    'forecast_baseline_summary_path': '../data/forecast_plus1_baselines_v10_enriched_yoy_corridors.csv',
    'forecast_baseline_details_path': '../data/forecast_plus1_baseline_details_v10_enriched_yoy_corridors.csv',
    'ablation_details_path': '../data/ablation_fidelity_details_v10_enriched_yoy_corridors.csv',
    'counterfactual_details_path': '../data/counterfactual_details_v10_enriched_yoy_corridors.csv',
    'grounding_details_path': '../data/grounding_details_v10_enriched_yoy_corridors.csv',
    'checkpoint_selection_path': '../data/forecast_checkpoint_scores_v10_enriched_yoy_corridors.csv',
    'best_checkpoint_path': '../data/best_forecast_checkpoint_v10_enriched_yoy_corridors.csv',
}

print('=== CONFIG ===')
for k, v in CONFIG.items():
    if k in ('hf_token', 'openai_api_key'):
        print(f'  {k:35s}: {"+ SET" if v else "x NOT SET"}')
    else:
        print(f'  {k:35s}: {v}')


=== CONFIG ===
  experiment_name                    : v10_enriched_yoy_corridors
  model_name                         : meta-llama/Meta-Llama-3.1-8B-Instruct
  hf_token                           : + SET
  openai_api_key                     : x NOT SET
  load_in_4bit                       : True
  load_in_8bit                       : False
  lora_r                             : 16
  lora_alpha                         : 32
  lora_dropout                       : 0.1
  target_modules                     : ['q_proj', 'v_proj', 'k_proj', 'o_proj']
  num_epochs                         : 3
  per_device_train_batch_size        : 4
  gradient_accumulation_steps        : 2
  learning_rate                      : 0.0002
  warmup_ratio                       : 0.05
  max_seq_length                     : 3072
  fp16                               : True
  temperature                        : 0.95
  max_new_tokens                     : 256
  init_adapter_path                  : None
  ablation_variant  

In [4]:
# ── 2. INSTALL PACKAGES ───────────────────────────────────────────────────────
import subprocess, sys

PACKAGES = [
    'transformers>=4.45.0,<5.0.0',
    'peft>=0.13.0',
    'trl>=0.11.0',
    'accelerate>=0.34.0',
    'bitsandbytes>=0.43.1',
    'datasets>=2.20.0',
    'huggingface_hub>=0.25.0',
    'scipy>=1.13.0',
    'openai>=1.0.0',
    'protobuf>=5.0.0',
    'sentencepiece>=0.2.0',
]

for pkg in PACKAGES:
    name = pkg.split('>=')[0].split('==')[0]
    try:
        __import__(name.replace('-', '_'))
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg],
                       check=True, capture_output=True)
        print(f'  ✓ {pkg} installed')


c:\Users\aiilab_kuro\Documents\Mathieu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ transformers>=4.45.0,<5.0.0
  ✓ peft>=0.13.0
  ✓ trl>=0.11.0
  ✓ accelerate>=0.34.0
  ✓ bitsandbytes>=0.43.1
  ✓ datasets>=2.20.0
  ✓ huggingface_hub>=0.25.0
  ✓ scipy>=1.13.0
  ✓ openai>=1.0.0
  Installing protobuf>=5.0.0...
  ✓ protobuf>=5.0.0 installed
  ✓ sentencepiece>=0.2.0


In [5]:
# ── 3. IMPORTS ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import gc, json, re, pathlib, time
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer.utils import DataCollatorForCompletionOnlyLM
    except ImportError:
        from trl.trainer import DataCollatorForCompletionOnlyLM
from datasets import Dataset

DATA_DIR = pathlib.Path('../data')
LLM_DIR  = pathlib.Path('../llm')
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print('All imports OK')
print(f'  transformers : {__import__("transformers").__version__}')
print(f'  peft         : {__import__("peft").__version__}')
print(f'  trl          : {__import__("trl").__version__}')
print(f'  torch        : {torch.__version__}')


All imports OK
  transformers : 4.57.6
  peft         : 0.18.1
  trl          : 0.8.6
  torch        : 2.5.1+cu121


In [6]:
# ── 4. LOAD + INSPECT DATA ────────────────────────────────────────────────────
records = []
with open(CONFIG['data_path'], 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df_prompts = pd.DataFrame(records)
print(f'Total prompts : {len(df_prompts)}')
print(f'Columns       : {list(df_prompts.columns)}')

print(f'\nSplit distribution:')
print(df_prompts['split'].value_counts().to_string())

print(f'\nQuestion types:')
print(df_prompts['question_type'].value_counts().to_string())

print(f'\nTraffic data: real monthly values (2023-2024, 24 timesets per corridor)')
print(f'  → TT ratio semantics: 1.0 = free-flow; higher = slower / more congested')
print(f'  → If prompts were regenerated, verify traffic_is_static=False and prompt count matches the active experiment design')

# Prompt length stats
df_prompts['prompt_len'] = df_prompts['prompt'].str.len()
print(f'\nPrompt length (chars):')
print(f'  mean={df_prompts["prompt_len"].mean():.0f}')
print(f'  max ={df_prompts["prompt_len"].max()}')
print(f'  min ={df_prompts["prompt_len"].min()}')
print(f'  Prompts > 3072 chars : {(df_prompts["prompt_len"] > 3072).sum()}')

train_sample = df_prompts[df_prompts['split'] == 'train']
if not train_sample.empty:
    sample_candidates = train_sample[train_sample['question_type'] == 'nowcast']
    sample_label = 'first train nowcast'
    if sample_candidates.empty:
        sample_candidates = train_sample[train_sample['question_type'] == 'forecast']
        sample_label = 'first train forecast'
    if sample_candidates.empty:
        sample_candidates = train_sample
        sample_label = 'first train prompt'
    print(f'\nSample prompt ({sample_label}):')
    sample = sample_candidates.iloc[0]
    print(sample['prompt'][:1500])
    print('...')
else:
    print('\nSample prompt: no train prompts available')


Total prompts : 528
Columns       : ['id', 'corridor_id', 'corr_id', 'year', 'month', 'season', 'split', 'question_type', 'prompt_variant', 'question', 'answer', 'prompt', 'has_full_history', 'traffic_is_static']

Split distribution:
split
train    240
val      144
test     144

Question types:
question_type
forecast    288
nowcast      48
explain      48
whatif       48
decision     48
tourism      48

Traffic data: real monthly values (2023-2024, 24 timesets per corridor)
  → TT ratio semantics: 1.0 = free-flow; higher = slower / more congested
  → If prompts were regenerated, verify traffic_is_static=False and prompt count matches the active experiment design

Prompt length (chars):
  mean=2154
  max =2991
  min =1179
  Prompts > 3072 chars : 0

Sample prompt (first train forecast):
[SYSTEM] You are an expert traffic analyst for Phuket island, Thailand. You have access only to the information shown in the prompt. Provide accurate, data-driven answers about traffic conditions, foreca

In [7]:
# ── 5. ABLATION MASKING ───────────────────────────────────────────────────────
# Retire des blocs du prompt pour entraîner les variantes d'ablation.
# Utilisé en inférence pour les tests explainability même sans re-entraînement.

BLOCK_RE = re.compile(
    r'(\[(?:SYSTEM|CORRIDOR|TRAFFIC PROFILE|CONTEXT|HISTORY|QUESTION|ANSWER)\])',
    re.MULTILINE
)

def parse_blocks(prompt_text):
    """Décompose le prompt en blocs ordonnés {header: contenu}."""
    parts = BLOCK_RE.split(prompt_text)
    blocks = {}
    current_key = None
    for part in parts:
        if BLOCK_RE.fullmatch(part):
            current_key = part
            blocks[current_key] = ''
        elif current_key is not None:
            blocks[current_key] += part
    return blocks

def reconstruct_prompt(blocks):
    return ''.join(f'{k}{v}' for k, v in blocks.items())

def remove_context_lines(ctx, keywords):
    """Retire les lignes [CONTEXT] contenant une famille de variables."""
    kws = [kw.lower() for kw in keywords]
    return '\n'.join(
        line for line in ctx.split('\n')
        if not any(kw in line.lower() for kw in kws)
    )

def strip_context_season_label(ctx):
    """Retire le label saisonnier de la ligne date pour éviter une fuite météo en v2."""
    lines = ctx.split('\n')
    if lines:
        # Exemple: ' 2024-06 — June 2024 (monsoon season)' -> ' 2024-06 — June 2024'
        lines[0] = re.sub(r'\s*\([^)]*season[^)]*\)', '', lines[0], flags=re.IGNORECASE)
    return '\n'.join(lines)

def mask_prompt_blocks(prompt_text, variant):
    """Masque les blocs selon la variante d'ablation."""
    if variant == 'full':
        return prompt_text

    blocks = parse_blocks(prompt_text)

    if variant == 'flow_only':
        # v1 : retire [CONTEXT] et [HISTORY]
        blocks.pop('[CONTEXT]', None)
        blocks.pop('[HISTORY]', None)

    elif variant == 'flow_cal':
        # v2 : garde traffic + calendar ; retire tourism, weather et social/trends.
        if '[CONTEXT]' in blocks:
            blocks['[CONTEXT]'] = remove_context_lines(
                strip_context_season_label(blocks['[CONTEXT]']),
                ['tourism', 'passengers', 'pax', 'arrivals', 'hkt',
                 'weather', 'rain', 'temperature', 'wind', 'precipitation', 'mm rain', 'km/h wind',
                 'trends', 'google', 'search interest']
            )
        blocks.pop('[HISTORY]', None)

    elif variant == 'flow_cal_wx':
        # v3 : garde traffic + calendar + weather ; retire tourism et social/trends.
        if '[CONTEXT]' in blocks:
            blocks['[CONTEXT]'] = remove_context_lines(
                blocks['[CONTEXT]'],
                ['tourism', 'passengers', 'pax', 'arrivals', 'hkt',
                 'trends', 'google', 'search interest']
            )
        # Garde [HISTORY]

    return reconstruct_prompt(blocks)


# Test masking
sample_prompt = df_prompts.iloc[0]['prompt']
for variant in ['full', 'flow_only', 'flow_cal', 'flow_cal_wx']:
    masked = mask_prompt_blocks(sample_prompt, variant)
    print(f'  {variant:15s}: {len(masked):4d} chars  (original: {len(sample_prompt):4d})')




  full           : 1722 chars  (original: 1722)
  flow_only      : 1202 chars  (original: 1722)
  flow_cal       : 1483 chars  (original: 1722)
  flow_cal_wx    : 1644 chars  (original: 1722)


In [8]:
# -- 6. PREPARE HUGGINGFACE DATASET -------------------------------------------
# Format SFT: {"text": prompt_complet}
# Le prompt contient déjà [SYSTEM]...[QUESTION]...[ANSWER]
# DataCollatorForCompletionOnlyLM calculera la loss uniquement sur [ANSWER]

variant = CONFIG['ablation_variant']

def prepare_sft_record(row):
    masked = mask_prompt_blocks(row['prompt'], variant)
    return {'text': masked, 'split': row['split'], 'question_type': row['question_type'],
            'prompt_variant': row.get('prompt_variant', 'full'),
            'corr_id': int(row['corr_id']), 'corridor_id': row['corridor_id'],
            'year': int(row['year']), 'month': int(row['month']),
            'answer': row['answer']}

df_sft = df_prompts.apply(prepare_sft_record, axis=1, result_type='expand')

train_records = df_sft[df_sft['split'] == 'train'].to_dict('records')
val_records   = df_sft[df_sft['split'] == 'val'].to_dict('records')
test_records  = df_sft[df_sft['split'] == 'test'].to_dict('records')

train_ds = Dataset.from_list(train_records)
val_ds   = Dataset.from_list(val_records)
test_ds  = Dataset.from_list(test_records)

print(f'Ablation variant : {variant}')
print(f'Train dataset    : {len(train_ds)} prompts')
print(f'Val dataset      : {len(val_ds)} prompts')
print(f'Test dataset     : {len(test_ds)} prompts')
print(f'\nTrain question types:')
train_df = pd.DataFrame(train_records)
print(train_df['question_type'].value_counts().to_string())
print(f'\nTrain prompt variants:')
print(train_df['prompt_variant'].value_counts().to_string())


Ablation variant : full
Train dataset    : 240 prompts
Val dataset      : 144 prompts
Test dataset     : 144 prompts

Train question types:
question_type
forecast    240

Train prompt variants:
prompt_variant
full                     48
forecast_traffic_core    48
forecast_lag_focus       48
forecast_profile_only    48
forecast_minimal_json    48


In [9]:
# ── 6b. ZERO-SHOT EVALUATION — ADAPTATION TABLE 5 DU PAPIER ──────────────────
# Papier xTP-LLM Table 5 : comparaison de LLMs sur des jeux non vus.
# Ici : comparaison zero-shot des LLMs bruts sur Phuket, tâche forecast +1 mois.
# Important : on évalue un vrai horizon futur, pas le nowcast dont la valeur est déjà dans le prompt.

ZERO_SHOT_MODELS = [
    # ── Identiques papier (Table 5 xTP-LLM) ───────────────────────────────────
    {'name': 'meta-llama/Llama-2-7b-chat-hf',          'label': 'Llama2-7B-chat',   'type': 'hf',     'min_vram_gb': 8},
    {'name': 'meta-llama/Llama-2-13b-chat-hf',         'label': 'Llama2-13B-chat',  'type': 'hf',     'min_vram_gb': 10},
    {'name': 'meta-llama/Llama-2-70b-chat-hf',         'label': 'Llama2-70B-chat',  'type': 'hf',     'min_vram_gb': 30},
    {'name': 'gpt-3.5-turbo',                          'label': 'GPT-3.5-turbo',    'type': 'openai', 'min_vram_gb': 0},
    {'name': 'gpt-4o',                                 'label': 'GPT-4o',           'type': 'openai', 'min_vram_gb': 0},
    # ── Nos extensions ────────────────────────────────────────────────────────
    {'name': 'meta-llama/Meta-Llama-3.1-8B-Instruct',  'label': 'Llama3.1-8B-ZS',  'type': 'hf',     'min_vram_gb': 8},
    {'name': 'mistralai/Mistral-7B-Instruct-v0.3',     'label': 'Mistral-7B-ZS',    'type': 'hf',     'min_vram_gb': 8},
]

_ZS_TARGET  = 'tt_ratio_Weekday_AM1'
_ZS_QTYPE   = 'forecast'
_ZS_HORIZON = 1
_all_meta_zs = pd.read_csv(CONFIG['dataset_csv'])

def _cuda_free_total_gb():
    if not torch.cuda.is_available():
        return 0.0, 0.0
    try:
        free_b, total_b = torch.cuda.mem_get_info()
    except Exception:
        total_b = torch.cuda.get_device_properties(0).total_memory
        reserved_b = torch.cuda.memory_reserved(0)
        free_b = max(total_b - reserved_b, 0)
    return free_b / 1e9, total_b / 1e9

_free_vram_gb, _total_vram_gb = _cuda_free_total_gb()

# ── Helpers (définis ici, avant Cell 7/11 qui redéfinissent les versions globales)
_BLOCK_RE_ZS = re.compile(
    r'(\[(?:SYSTEM|CORRIDOR|TRAFFIC PROFILE|CONTEXT|HISTORY|QUESTION|ANSWER)\])',
    re.MULTILINE,
)

def _parse_blocks_zs(prompt_text):
    parts = _BLOCK_RE_ZS.split(prompt_text)
    blocks, cur = {}, None
    for part in parts:
        if _BLOCK_RE_ZS.fullmatch(part):
            cur = part; blocks[cur] = ''
        elif cur is not None:
            blocks[cur] += part
    return blocks

def _add_month(year, month, horizon=1):
    idx = (int(year) * 12 + int(month) - 1) + horizon
    return idx // 12, idx % 12 + 1

def _true_future_value(row, meta_df, target, horizon=1):
    y, m = _add_month(row['year'], row['month'], horizon)
    mask = (
        (meta_df['corr_id'] == int(row['corr_id'])) &
        (meta_df['year']    == y) &
        (meta_df['month']   == m)
    )
    if mask.sum() == 0:
        return None
    return float(meta_df.loc[mask, target].iloc[0])

def _add_forecast_eval_instruction(prompt_text):
    """Ajoute une contrainte de sortie avant [ANSWER], sans exposer la réponse cible."""
    instr = (
        '\n[NUMERIC OUTPUT]\n'
        'Predict the next calendar month AM Peak travel-time ratio. '
        'Begin your answer exactly with: Next-month AM TT ratio: <number>.\n'
    )
    idx = prompt_text.find('[ANSWER]')
    if idx == -1:
        return prompt_text + instr + '\n[ANSWER]'
    return prompt_text[:idx] + instr + prompt_text[idx:]

def _extract_system_user(prompt_text):
    """Sépare [SYSTEM] du reste pour chat APIs."""
    prompt_text = _add_forecast_eval_instruction(prompt_text)
    blocks = _parse_blocks_zs(prompt_text)
    system = blocks.get('[SYSTEM]', '').strip()
    user_parts = [
        f'{k}{blocks[k]}'
        for k in ['[CORRIDOR]', '[TRAFFIC PROFILE]', '[CONTEXT]', '[HISTORY]', '[QUESTION]']
        if k in blocks
    ]
    return system, '\n'.join(user_parts)

def _extract_numeric_zs(text):
    """Extrait un tt_ratio plausible, en priorisant les patterns explicitement AM/next-month."""
    patterns = [
        r'next[- ]month[^\n]{0,80}?(?:AM|morning)?[^\n]{0,80}?(?:TT ratio|tt_ratio|travel[- ]time ratio)?[^\n]{0,40}?(\d+\.\d{2,4})',
        r'(?:AM Peak|AM|morning)[^\n]{0,80}?(?:TT ratio|tt_ratio|travel[- ]time ratio)[^\n]{0,40}?(\d+\.\d{2,4})',
        r'(?:TT ratio|tt_ratio|travel[- ]time ratio)[^\n]{0,40}?(\d+\.\d{2,4})',
        r'\b([012]\.\d{2,4})\b',
    ]
    for pat in patterns:
        for m in re.findall(pat, text, flags=re.IGNORECASE):
            v = float(m)
            if 0.5 <= v <= 3.0:
                return v
    return None

def _metrics_zs(trues, preds, label, n_eligible):
    if not preds:
        return None
    t, p = np.array(trues), np.array(preds)
    mae  = float(np.mean(np.abs(p - t)))
    rmse = float(np.sqrt(np.mean((p - t) ** 2)))
    nz   = np.abs(t) > 1e-10
    mape = float(np.mean(np.abs((t[nz] - p[nz]) / t[nz])) * 100) if nz.sum() > 0 else float('nan')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape,
            'parse_rate': len(preds) / max(n_eligible, 1) * 100, 'n': len(preds),
            'n_eligible': n_eligible, 'question_type': _ZS_QTYPE, 'horizon_months': _ZS_HORIZON}

# ── Évaluation OpenAI (GPT-3.5-turbo, GPT-4o) ────────────────────────────────
def _eval_openai_zs(model_name, label, prompts):
    import openai
    api_key = CONFIG.get('openai_api_key') or os.getenv('OPENAI_API_KEY')
    if not api_key:
        print(f'    SKIP — OPENAI_API_KEY not set')
        return None
    client  = openai.OpenAI(api_key=api_key)
    preds, trues, n_eligible = [], [], 0
    for row in prompts:
        true_val = _true_future_value(row, _all_meta_zs, _ZS_TARGET, _ZS_HORIZON)
        if true_val is None:
            continue
        n_eligible += 1
        system_text, user_text = _extract_system_user(row['text'])
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': system_text[:2000]},
                    {'role': 'user',   'content': user_text[:6000] + '\n[ANSWER]'},
                ],
                max_tokens=256,
                temperature=0.0,
            )
            generated = resp.choices[0].message.content.strip()
        except Exception as e:
            print(f'    API error: {e}')
            continue
        pred = _extract_numeric_zs(generated)
        if pred is None:
            continue
        preds.append(pred)
        trues.append(true_val)
    return _metrics_zs(trues, preds, label, n_eligible)

# ── Évaluation HF (Llama2, Llama3.1, Mistral) ────────────────────────────────
def _eval_hf_zs(model_name, label, prompts, hf_token):
    """Charge le modèle en 4-bit, évalue en zero-shot, libère VRAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        tok_zs = AutoTokenizer.from_pretrained(model_name, token=hf_token, trust_remote_code=True)
        tok_zs.pad_token = tok_zs.eos_token
        tok_zs.padding_side = 'right'
        mdl_zs = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
            ),
            device_map='auto',
            token=hf_token,
            trust_remote_code=True,
        )
        mdl_zs.config.use_cache = False
        # Avoid noisy generation warnings from chat-model generation configs.
        mdl_zs.generation_config.do_sample = False
        mdl_zs.generation_config.temperature = None
        mdl_zs.generation_config.top_p = None
        mdl_zs.generation_config.max_length = 20
    except Exception as e:
        print(f'    Load error: {e}')
        return None

    preds, trues, n_eligible = [], [], 0
    for row in prompts:
        true_val = _true_future_value(row, _all_meta_zs, _ZS_TARGET, _ZS_HORIZON)
        if true_val is None:
            continue
        n_eligible += 1
        system_text, user_text = _extract_system_user(row['text'])
        try:
            messages   = [{'role': 'system', 'content': system_text},
                          {'role': 'user',   'content': user_text}]
            input_text = tok_zs.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            input_text = f'{system_text}\n\n{user_text}\n[ANSWER]'

        inputs = tok_zs(input_text, return_tensors='pt', truncation=True,
                        max_length=3072).to(mdl_zs.device)
        in_len = inputs['input_ids'].shape[1]
        with torch.no_grad():
            out = mdl_zs.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tok_zs.eos_token_id,
            )
        generated = tok_zs.decode(out[0][in_len:], skip_special_tokens=True).strip()
        pred = _extract_numeric_zs(generated)
        if pred is None:
            continue
        preds.append(pred)
        trues.append(true_val)

    del mdl_zs, tok_zs
    gc.collect()
    torch.cuda.empty_cache()
    return _metrics_zs(trues, preds, label, n_eligible)

# ── Lancer l'évaluation ───────────────────────────────────────────────────────
_val_forecast = [r for r in val_records if r['question_type'] == _ZS_QTYPE]
print(f'Zero-shot eval | {len(_val_forecast)} val forecast prompts | horizon=+{_ZS_HORIZON} month | target={_ZS_TARGET}')
print(f'GPU VRAM total : {_total_vram_gb:.1f} GB')
print(f'GPU VRAM free  : {_free_vram_gb:.1f} GB\n')

ZERO_SHOT_RESULTS = {}  # Partagé avec Cell 17 (final summary)

for m_cfg in ZERO_SHOT_MODELS:
    lbl, mtype, mname = m_cfg['label'], m_cfg['type'], m_cfg['name']
    min_vram = m_cfg.get('min_vram_gb', 0)
    print(f'── {lbl} ', end='', flush=True)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _free_vram_gb, _total_vram_gb = _cuda_free_total_gb()

    if mtype == 'openai':
        r = _eval_openai_zs(mname, lbl, _val_forecast)
    elif mtype == 'hf':
        if _free_vram_gb < min_vram:
            print(f'SKIP (free VRAM {_free_vram_gb:.1f}GB < {min_vram}GB needed)')
            ZERO_SHOT_RESULTS[lbl] = None
            continue
        r = _eval_hf_zs(mname, lbl, _val_forecast, CONFIG.get('hf_token'))
    else:
        r = None

    ZERO_SHOT_RESULTS[lbl] = r
    if r:
        print(f'→ MAE={r["MAE"]:.6f}  RMSE={r["RMSE"]:.6f}  MAPE={r["MAPE"]:.4f}%  (parse {r["parse_rate"]:.0f}%, n={r["n"]}/{r["n_eligible"]})')
    else:
        print('→ No result (skipped or failed)')

# ── Tableau intermédiaire ─────────────────────────────────────────────────────
print(f'\n{"─"*82}')
print(f'{"Model":<28} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10} {"parse%":>8} {"n":>8}')
print(f'{"─"*28} {"─"*10} {"─"*10} {"─"*10} {"─"*8} {"─"*8}')
for lbl, r in ZERO_SHOT_RESULTS.items():
    if r:
        print(f'{lbl:<28} {r["MAE"]:>10.6f} {r["RMSE"]:>10.6f} {r["MAPE"]:>10.4f} {r["parse_rate"]:>7.1f}% {r["n"]:>3}/{r["n_eligible"]:<3}')
    else:
        print(f'{lbl:<28} {"SKIP/FAIL":>10}')
print(f'{"─"*82}')
print('→ Ces résultats sont des zero-shot bruts sur Phuket forecast +1 mois, pas les valeurs originales CATraffic du papier.')


Zero-shot eval | 24 val forecast prompts | horizon=+1 month | target=tt_ratio_Weekday_AM1
GPU VRAM total : 17.2 GB
GPU VRAM free  : 15.8 GB

── Llama2-7B-chat 

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.45s/it]


→ MAE=0.171458  RMSE=0.237770  MAPE=11.7450%  (parse 100%, n=24/24)
── Llama2-13B-chat SKIP (free VRAM 9.1GB < 10GB needed)
── Llama2-70B-chat SKIP (free VRAM 9.1GB < 30GB needed)
── GPT-3.5-turbo     SKIP — OPENAI_API_KEY not set
→ No result (skipped or failed)
── GPT-4o     SKIP — OPENAI_API_KEY not set
→ No result (skipped or failed)
── Llama3.1-8B-ZS 

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.93s/it]


→ MAE=0.136375  RMSE=0.175617  MAPE=9.4533%  (parse 100%, n=24/24)
── Mistral-7B-ZS 

Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.59s/it]


→ MAE=0.134583  RMSE=0.172373  MAPE=9.4697%  (parse 100%, n=24/24)

──────────────────────────────────────────────────────────────────────────────────
Model                               MAE       RMSE    MAPE(%)   parse%        n
──────────────────────────── ────────── ────────── ────────── ──────── ────────
Llama2-7B-chat                 0.171458   0.237770    11.7450   100.0%  24/24 
Llama2-13B-chat               SKIP/FAIL
Llama2-70B-chat               SKIP/FAIL
GPT-3.5-turbo                 SKIP/FAIL
GPT-4o                        SKIP/FAIL
Llama3.1-8B-ZS                 0.136375   0.175617     9.4533   100.0%  24/24 
Mistral-7B-ZS                  0.134583   0.172373     9.4697   100.0%  24/24 
──────────────────────────────────────────────────────────────────────────────────
→ Ces résultats sont des zero-shot bruts sur Phuket forecast +1 mois, pas les valeurs originales CATraffic du papier.


In [10]:
# ── 6c. SAVE ZERO-SHOT RESULTS IMMEDIATELY ────────────────────────────────────
# À lancer juste après 6b : sauvegarde les baselines zero-shot sans attendre la fin du notebook.

if 'ZERO_SHOT_RESULTS' not in globals():
    raise RuntimeError('ZERO_SHOT_RESULTS not found — run cell 6b first.')

zs_rows = []
for lbl, res in ZERO_SHOT_RESULTS.items():
    if res is None:
        zs_rows.append({
            'label': lbl,
            'status': 'SKIP/FAIL',
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'parse_rate': np.nan,
            'n': 0,
            'n_eligible': np.nan,
            'question_type': 'forecast',
            'horizon_months': 1,
        })
    else:
        zs_rows.append({**res, 'status': 'OK'})

zs_df = pd.DataFrame(zs_rows)
zs_df.to_csv(CONFIG['zs_path'], index=False)
print(f'Zero-shot results saved: {CONFIG["zs_path"]}')
print(zs_df.to_string(index=False))


Zero-shot results saved: ../llm/outputs/zero_shot_results_v10_enriched_yoy_corridors.csv
          label      MAE     RMSE      MAPE  parse_rate  n  n_eligible question_type  horizon_months    status
 Llama2-7B-chat 0.171458 0.237770 11.744998       100.0 24        24.0      forecast               1        OK
Llama2-13B-chat      NaN      NaN       NaN         NaN  0         NaN      forecast               1 SKIP/FAIL
Llama2-70B-chat      NaN      NaN       NaN         NaN  0         NaN      forecast               1 SKIP/FAIL
  GPT-3.5-turbo      NaN      NaN       NaN         NaN  0         NaN      forecast               1 SKIP/FAIL
         GPT-4o      NaN      NaN       NaN         NaN  0         NaN      forecast               1 SKIP/FAIL
 Llama3.1-8B-ZS 0.136375 0.175617  9.453344       100.0 24        24.0      forecast               1        OK
  Mistral-7B-ZS 0.134583 0.172373  9.469711       100.0 24        24.0      forecast               1        OK


In [11]:
# ── 7. LOAD MODEL + TOKENIZER ─────────────────────────────────────────────────
print(f'Loading model: {CONFIG["model_name"]}')
print(f'Quantization : {"4-bit QLoRA" if CONFIG["load_in_4bit"] else "8-bit" if CONFIG["load_in_8bit"] else "FP16"}')

# BitsAndBytes config
if CONFIG['load_in_4bit']:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',           # NF4 — optimal pour QLoRA
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,      # Double quantization — économise ~0.4 bit/param
    )
elif CONFIG['load_in_8bit']:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)  # Réplique exacte du papier
else:
    bnb_config = None  # FP16 full precision

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['model_name'],
    token=CONFIG['hf_token'],
    trust_remote_code=True,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'   # Important pour éviter les warnings avec Llama

# Modèle
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    token=CONFIG['hf_token'],
    trust_remote_code=True,
)
model.config.use_cache            = False
model.config.pretraining_tp       = 1
model.config.pad_token_id         = tokenizer.eos_token_id

print(f'\nModel loaded!')
print(f'  Device map : {model.hf_device_map if hasattr(model, "hf_device_map") else "auto"}')
total_params = sum(p.numel() for p in model.parameters())
print(f'  Total params : {total_params/1e9:.2f}B')

# VRAM usage after loading
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated(0) / 1e9
    print(f'  VRAM used : {vram_used:.1f} GB')

Loading model: meta-llama/Meta-Llama-3.1-8B-Instruct
Quantization : 4-bit QLoRA


Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.90s/it]



Model loaded!
  Device map : {'': 0}
  Total params : 4.54B
  VRAM used : 5.7 GB


In [12]:
# ── 8. LORA SETUP ─────────────────────────────────────────────────────────────
# V7: warm-start from the saved V4 adapter when available, otherwise create a
# fresh LoRA adapter with the same topology.

# Préparer le modèle pour l'entraînement kbit (gradient checkpointing, cast non-quant layers)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

init_adapter_path = CONFIG.get('init_adapter_path')
if init_adapter_path:
    adapter_path = pathlib.Path(init_adapter_path)
    if not adapter_path.exists():
        raise FileNotFoundError(f'Warm-start adapter not found: {init_adapter_path}')
    print(f'Warm-start adapter: {init_adapter_path}')
    model = PeftModel.from_pretrained(model, init_adapter_path, is_trainable=True)
    loaded_cfg = model.peft_config.get('default') if hasattr(model, 'peft_config') else None
else:
    lora_config = LoraConfig(
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        target_modules=CONFIG['target_modules'],   # attention projection layers
        lora_dropout=CONFIG['lora_dropout'],
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)
    loaded_cfg = lora_config

# Stats trainable params
trainable, total = 0, 0
for _, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()
print(f'Trainable params : {trainable:,}  ({100*trainable/total:.3f}% of total)')
print(f'Total params     : {total:,}')
print(f'\nLoRA config:')
print(f'  r             = {getattr(loaded_cfg, "r", CONFIG["lora_r"])}')
print(f'  alpha         = {getattr(loaded_cfg, "lora_alpha", CONFIG["lora_alpha"])}')
print(f'  target_modules= {getattr(loaded_cfg, "target_modules", CONFIG["target_modules"])}')
print(f'  dropout       = {getattr(loaded_cfg, "lora_dropout", CONFIG["lora_dropout"])}')

Trainable params : 13,631,488  (0.299% of total)
Total params     : 4,554,231,808

LoRA config:
  r             = 16
  alpha         = 32
  target_modules= {'o_proj', 'v_proj', 'q_proj', 'k_proj'}
  dropout       = 0.1


In [13]:
# ── 9. SFTTRAINER — ENTRAÎNEMENT ──────────────────────────────────────────────
# V8: forecast-pure from-scratch LoRA.
# Goal: maximize next-month forecast signal without inheriting V4 forecast anchors.
# Différence : loss calculée UNIQUEMENT sur [ANSWER] (completion-only)

# DataCollator pour loss sur [ANSWER] uniquement (Eq. 3 du papier)
response_template = '[ANSWER]'
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)


training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    fp16=CONFIG['fp16'],
    bf16=False,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    save_total_limit=2,
    report_to='none',   # pas de W&B
    optim='paged_adamw_32bit',  # optimal pour QLoRA
    lr_scheduler_type='cosine',
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_length'],
    data_collator=collator,
    packing=False,
    args=training_args,
)

print(f'Training {len(train_ds)} samples × {CONFIG["num_epochs"]} epochs')
print(f'Effective batch size: {CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]}')
print(f'Estimated steps per epoch: {len(train_ds) // (CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"])}')
print('\nStarting training...')
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print(f'\nTraining complete in {elapsed/60:.1f} min')
print(f'Final train loss : {train_result.training_loss:.4f}')

# Log metrics
print('\nTraining log (loss per step):')
for log in trainer.state.log_history:
    if 'loss' in log and 'epoch' in log:
        print(f'  step={log.get("step","?"):4d}  epoch={log["epoch"]:.2f}  loss={log["loss"]:.4f}')


Map: 100%|██████████| 144/144 [00:00<00:00, 2649.45 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Training 240 samples × 3 epochs
Effective batch size: 8
Estimated steps per epoch: 30

Starting training...


Epoch,Training Loss,Validation Loss
1,0.162000,2.257391
2,0.069400,2.477568
3,0.017100,2.635407



Training complete in 11.1 min
Final train loss : 0.1732

Training log (loss per step):
  step=   5  epoch=0.17  loss=1.3172
  step=  10  epoch=0.33  loss=0.3043
  step=  15  epoch=0.50  loss=0.2096
  step=  20  epoch=0.67  loss=0.1859
  step=  25  epoch=0.83  loss=0.1962
  step=  30  epoch=1.00  loss=0.1620
  step=  35  epoch=1.17  loss=0.1336
  step=  40  epoch=1.33  loss=0.1190
  step=  45  epoch=1.50  loss=0.0967
  step=  50  epoch=1.67  loss=0.0878
  step=  55  epoch=1.83  loss=0.0927
  step=  60  epoch=2.00  loss=0.0694
  step=  65  epoch=2.17  loss=0.0293
  step=  70  epoch=2.33  loss=0.0409
  step=  75  epoch=2.50  loss=0.0195
  step=  80  epoch=2.67  loss=0.0216
  step=  85  epoch=2.83  loss=0.0147
  step=  90  epoch=3.00  loss=0.0171


In [14]:
# ── 10. SAVE ADAPTER + LOSS CURVES ────────────────────────────────────────────
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'Adapter saved: {CONFIG["output_dir"]}')

# Loss curves
logs     = trainer.state.log_history
steps    = [l['step']       for l in logs if 'loss' in l]
tr_loss  = [l['loss']       for l in logs if 'loss' in l]
val_logs = [l for l in logs if 'eval_loss' in l]

if val_logs:
    val_steps = [l['step'] for l in val_logs]
    val_loss  = [l['eval_loss'] for l in val_logs]
    print(f'\nValidation loss per epoch:')
    for s, v in zip(val_steps, val_loss):
        print(f'  step={s}  eval_loss={v:.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, tr_loss, label='Train loss', alpha=0.7)
if val_logs:
    ax.plot(val_steps, val_loss, 'r-o', label='Val loss', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Curves')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
loss_path = DATA_DIR / f'llm_loss_curves_{CONFIG["experiment_name"]}.png'
plt.savefig(loss_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nLoss curve saved: {loss_path}')

Adapter saved: ../llm/outputs/lora_v10_enriched_yoy_corridors

Validation loss per epoch:
  step=30  eval_loss=2.2574
  step=60  eval_loss=2.4776
  step=90  eval_loss=2.6354

Loss curve saved: ..\data\llm_loss_curves_v10_enriched_yoy_corridors.png


In [15]:
# ── 11. INFÉRENCE — HELPERS ───────────────────────────────────────────────────

def get_question_prompt(prompt_text):
    """Retourne le prompt SANS la réponse (up to [ANSWER]) pour l'inférence."""
    idx = prompt_text.rfind('[ANSWER]')
    if idx != -1:
        return prompt_text[:idx + len('[ANSWER]')]
    return prompt_text


def add_forecast_eval_instruction(prompt_text):
    """Ajoute une consigne numérique strictement forecast avant [ANSWER]."""
    instr = (
        '\n[NUMERIC OUTPUT]\n'
        'Return JSON only on the first line in exactly this shape: '
        '{"next_month_am_tt_ratio": <number>, "delta_vs_current": <signed_number>}. '
        'Do not mention the current AM / PM / weekend TT ratios before or instead of that JSON line. '
        'After the JSON, add at most one short sentence using only information visible in the prompt.\n'
    )
    idx = prompt_text.rfind('[ANSWER]')
    if idx == -1:
        return prompt_text + instr + '\n[ANSWER]'
    return prompt_text[:idx] + instr + prompt_text[idx:]


@torch.no_grad()
def generate_response(prompt_text, max_new_tokens=None, deterministic=False, temperature=None):
    """Génère une réponse du modèle fine-tuné.

    deterministic=True est recommandé pour les métriques reproductibles.
    deterministic=False garde le sampling pour les démonstrations qualitatives.
    """
    if max_new_tokens is None:
        max_new_tokens = CONFIG['max_new_tokens']
    input_text   = get_question_prompt(prompt_text)
    inputs       = tokenizer(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=CONFIG['max_seq_length'],
    ).to(model.device)
    input_length = inputs['input_ids'].shape[1]

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
    )
    if deterministic:
        gen_kwargs['do_sample'] = False
    else:
        gen_kwargs['do_sample'] = True
        gen_kwargs['temperature'] = CONFIG['temperature'] if temperature is None else temperature

    outputs = model.generate(**gen_kwargs)
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated.strip()


def extract_numeric(text):
    """Extrait un tt_ratio plausible, en priorisant le JSON canonicalisé V9."""
    patterns = [
        r'"next_month_am_tt_ratio"\s*:\s*([012]\.\d{2,4})',
        r'next[-_ ]month[^\n]{0,80}?(?:AM|morning)?[^\n]{0,80}?([012]\.\d{2,4})',
        r'(?:AM Peak|AM|morning)[^\n]{0,80}?(?:TT ratio|tt_ratio|travel[- ]time ratio)[^\n]{0,40}?([012]\.\d{2,4})',
        r'(?:TT ratio|tt_ratio|travel[- ]time ratio)[^\n]{0,40}?([012]\.\d{2,4})',
        r'\b([012]\.\d{2,4})\b',
    ]
    for pat in patterns:
        for m in re.findall(pat, text, flags=re.IGNORECASE):
            v = float(m)
            if 0.5 <= v <= 3.0:
                return v
    return None

print('Inference helpers defined.')


Inference helpers defined.


In [16]:
# ── 12. INFÉRENCE — 1 EXEMPLE PAR QUESTION TYPE ───────────────────────────────
# Après fine-tuning, on évite les few-shot externes pour mesurer le modèle tel qu'il a été entraîné.

QUESTION_TYPES = ['nowcast', 'forecast', 'explain', 'whatif', 'decision', 'tourism']

print('=== INFÉRENCE — 1 exemple par question type (val set) ===\n')
inference_results = []

for qtype in QUESTION_TYPES:
    sample_rows = [r for r in val_records if r['question_type'] == qtype]
    if not sample_rows:
        print(f'  [{qtype}] No val samples found.')
        continue
    row = sample_rows[0]

    prompt_for_generation = add_forecast_eval_instruction(row['text']) if qtype == 'forecast' else row['text']
    generated = generate_response(prompt_for_generation, deterministic=True)
    pred_val = extract_numeric(generated)

    print(f'--- [{qtype.upper()}] corridor={row["corridor_id"]} | {row["year"]}-{row["month"]:02d} ---')
    print('Expected answer (first 300 chars):')
    print(f'  {row["answer"][:300]}')
    print('Generated answer:')
    print(f'  {generated[:300]}')
    if pred_val is not None:
        print(f'Extracted numeric prediction: {pred_val}')
    print()

    inference_results.append({
        'question_type': qtype,
        'corridor_id': row['corridor_id'],
        'year': row['year'],
        'month': row['month'],
        'generated': generated,
        'expected': row['answer'],
        'pred_numeric': pred_val,
    })


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== INFÉRENCE — 1 exemple par question type (val set) ===

--- [NOWCAST] corridor=Airport Road (Route 402) | 2024-01 ---
Expected answer (first 300 chars):
  Traffic conditions on Airport Road (Route 402) - January 2024 (high-season peak (Dec-Mar: dry weather, maximum tourist influx)):

Travel Time Ratios (TomTom - 1.0=free-flow, higher=slower/more congested):
  AM Peak  (Mon-Fri 07h): 1.260 [moderate] | Speed: 42.7 km/h
  PM Peak  (Mon-Fri 17h): 1.570 [
Generated answer:
  This month (January 2024), the current traffic conditions on Airport Road (Route 402) are experiencing a moderate level of congestion. According to the data, the current TT ratio is 1.320, indicating a 32% increase in travel time compared to free-flow conditions. The current speed on the road is app
Extracted numeric prediction: 1.32

--- [FORECAST] corridor=Airport Road (Route 402) | 2024-01 ---
Expected answer (first 300 chars):
  {"next_month_am_tt_ratio": 1.250, "delta_vs_current": -0.010}
Generated answer:
  {"

In [17]:
# ── 13. ÉVALUATION NUMÉRIQUE (MAE/RMSE/MAPE) ─────────────────────────────────
# V9 :
#   1. branche scratch sans warm-start
#   2. train forecast-only avec traffic-memory prompts
#   3. sélection du meilleur checkpoint sur VAL forecast MAE
#   4. sortie canonique JSON pour rapprocher le problème d'une régression structurée

PRIMARY_TARGET = 'tt_ratio_Weekday_AM1'
EVAL_QTYPE     = 'forecast'
EVAL_HORIZON   = 1
all_meta  = pd.read_csv(CONFIG['dataset_csv'])
val_meta  = pd.read_csv(CONFIG['val_csv'])
test_meta = pd.read_csv(CONFIG['test_csv'])


def add_month(year, month, horizon=1):
    idx = (int(year) * 12 + int(month) - 1) + horizon
    return idx // 12, idx % 12 + 1


def get_future_truth(row, meta_df, target=PRIMARY_TARGET, horizon=EVAL_HORIZON):
    y, m = add_month(row['year'], row['month'], horizon)
    mask = (
        (meta_df['corr_id'] == int(row['corr_id'])) &
        (meta_df['year']    == y) &
        (meta_df['month']   == m)
    )
    if mask.sum() == 0:
        return None, y, m, None
    true_val = float(meta_df.loc[mask, target].iloc[0])
    return true_val, y, m, meta_df.loc[mask].iloc[0]


def mape_safe(y_true, y_pred):
    mask = np.abs(np.array(y_true)) > 1e-10
    if mask.sum() == 0:
        return np.nan
    yt, yp = np.array(y_true)[mask], np.array(y_pred)[mask]
    return float(np.mean(np.abs((yt - yp) / yt)) * 100)


def metrics_from_pairs(trues, preds):
    mae  = mean_absolute_error(trues, preds)
    rmse = float(np.sqrt(mean_squared_error(trues, preds)))
    mape = mape_safe(trues, preds)
    return mae, rmse, mape


def _model_device(active_model):
    try:
        return next(active_model.parameters()).device
    except StopIteration:
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


@torch.no_grad()
def generate_response_for_model(active_model, prompt_text, max_new_tokens=64):
    input_text = get_question_prompt(prompt_text)
    device = _model_device(active_model)
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=CONFIG['max_seq_length'],
    ).to(device)
    input_length = inputs['input_ids'].shape[1]
    outputs = active_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated.strip()


def evaluate_split(records, split_name, active_model, verbose=True):
    eval_rows = [r for r in records if r['question_type'] == EVAL_QTYPE]
    if verbose:
        print(f'\n=== Évaluation numérique [{split_name}] — {len(eval_rows)} forecast prompts | horizon +{EVAL_HORIZON} mois ===')

    preds, trues, row_results = [], [], []
    skipped_no_truth = 0
    for row in eval_rows:
        true_val, target_year, target_month, target_meta = get_future_truth(row, all_meta)
        if true_val is None:
            skipped_no_truth += 1
            continue

        prompt_eval = add_forecast_eval_instruction(row['text'])
        generated = generate_response_for_model(active_model, prompt_eval, max_new_tokens=64)
        pred_val = extract_numeric(generated)

        base_info = {
            'split': split_name,
            'corr_id': int(row['corr_id']),
            'corridor_id': row['corridor_id'],
            'source_year': int(row['year']),
            'source_month': int(row['month']),
            'target_year': int(target_year),
            'target_month': int(target_month),
            'season': target_meta['season'] if target_meta is not None and 'season' in target_meta else None,
            'true': true_val,
            'generated': generated,
        }

        if pred_val is not None:
            preds.append(pred_val)
            trues.append(true_val)
            row_results.append({**base_info, 'pred': pred_val, 'abs_error': abs(true_val - pred_val), 'parsed': True})
            if verbose:
                print(f'  {str(row["corridor_id"]):20s} {row["year"]}-{row["month"]:02d} → {target_year}-{target_month:02d}  true={true_val:.4f}  pred={pred_val:.4f}  diff={abs(true_val-pred_val):.4f}')
        else:
            row_results.append({**base_info, 'pred': np.nan, 'abs_error': np.nan, 'parsed': False})
            if verbose:
                print(f'  {str(row["corridor_id"]):20s} {row["year"]}-{row["month"]:02d} → {target_year}-{target_month:02d}  true={true_val:.4f}  pred=NONE (parse failed)')

    n_eligible = len(eval_rows) - skipped_no_truth
    if len(preds) == 0:
        if verbose:
            print('  No parseable predictions — check extract_numeric()')
            print(f'  skipped_no_future_truth = {skipped_no_truth}')
        return None, pd.DataFrame(row_results)

    mae, rmse, mape = metrics_from_pairs(trues, preds)
    parse_rate = len(preds) / max(n_eligible, 1) * 100

    if verbose:
        print(f'\n  n_parsed = {len(preds)}/{n_eligible} ({parse_rate:.1f}%)')
        print(f'  skipped_no_future_truth = {skipped_no_truth}')
        print(f'  MAE  = {mae:.6f}')
        print(f'  RMSE = {rmse:.6f}')
        print(f'  MAPE = {mape:.4f}%')

    return {
        'split': split_name,
        'question_type': EVAL_QTYPE,
        'horizon_months': EVAL_HORIZON,
        'n': len(preds),
        'n_eligible': n_eligible,
        'parse_rate': parse_rate,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
    }, pd.DataFrame(row_results)


def checkpoint_step(path_obj):
    m = re.search(r'checkpoint-(\d+)$', path_obj.name)
    return int(m.group(1)) if m else -1


def load_adapter_candidate(checkpoint_dir):
    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG['model_name'],
        quantization_config=bnb_config,
        device_map='auto',
        token=CONFIG['hf_token'],
        trust_remote_code=True,
    )
    base_model.config.use_cache = False
    base_model.config.pretraining_tp = 1
    base_model.config.pad_token_id = tokenizer.eos_token_id
    adapted_model = PeftModel.from_pretrained(base_model, checkpoint_dir)
    adapted_model.eval()
    return adapted_model


checkpoint_dirs = sorted(
    [d for d in pathlib.Path(CONFIG['output_dir']).glob('checkpoint-*') if d.is_dir()],
    key=checkpoint_step,
)
checkpoint_selection_rows = []
BEST_FORECAST_CHECKPOINT = CONFIG['output_dir']
BEST_FORECAST_CHECKPOINT_INFO = None

if checkpoint_dirs:
    print('\n=== Checkpoint selection — validation forecast MAE ===')
    if 'model' in globals():
        del model
        gc.collect()
        torch.cuda.empty_cache()

    for ckpt in checkpoint_dirs:
        print(f'  Evaluating {ckpt.name} ...')
        cand_model = load_adapter_candidate(str(ckpt))
        cand_metrics, _ = evaluate_split(val_records, f'val@{ckpt.name}', cand_model, verbose=False)
        if cand_metrics is not None:
            checkpoint_selection_rows.append({
                'checkpoint': str(ckpt),
                'checkpoint_name': ckpt.name,
                'MAE': cand_metrics['MAE'],
                'RMSE': cand_metrics['RMSE'],
                'MAPE': cand_metrics['MAPE'],
                'parse_rate': cand_metrics['parse_rate'],
                'n': cand_metrics['n'],
            })
            print(f'    -> MAE={cand_metrics["MAE"]:.6f}  RMSE={cand_metrics["RMSE"]:.6f}  MAPE={cand_metrics["MAPE"]:.4f}%')
        else:
            checkpoint_selection_rows.append({
                'checkpoint': str(ckpt),
                'checkpoint_name': ckpt.name,
                'MAE': np.nan,
                'RMSE': np.nan,
                'MAPE': np.nan,
                'parse_rate': 0.0,
                'n': 0,
            })
            print('    -> no parseable predictions')
        del cand_model
        gc.collect()
        torch.cuda.empty_cache()

    checkpoint_selection_df = pd.DataFrame(checkpoint_selection_rows)
    valid_ckpts = checkpoint_selection_df.dropna(subset=['MAE']).sort_values('MAE')
    if len(valid_ckpts):
        best_row = valid_ckpts.iloc[0].to_dict()
        BEST_FORECAST_CHECKPOINT = best_row['checkpoint']
        BEST_FORECAST_CHECKPOINT_INFO = best_row
        print(f'\nBest checkpoint by val forecast MAE: {best_row["checkpoint_name"]}  (MAE={best_row["MAE"]:.6f})')
    else:
        BEST_FORECAST_CHECKPOINT = str(checkpoint_dirs[-1])
        BEST_FORECAST_CHECKPOINT_INFO = {'checkpoint': BEST_FORECAST_CHECKPOINT, 'checkpoint_name': checkpoint_dirs[-1].name}
        print(f'\nNo valid checkpoint metric found — fallback to latest checkpoint {checkpoint_dirs[-1].name}')

    model = load_adapter_candidate(BEST_FORECAST_CHECKPOINT)
    model.eval()
    model.save_pretrained(CONFIG['output_dir'])
    tokenizer.save_pretrained(CONFIG['output_dir'])
    print(f'Selected forecast-best adapter resaved to {CONFIG["output_dir"]}')
else:
    print('\nNo epoch checkpoints found — using current model in memory.')
    checkpoint_selection_df = pd.DataFrame()
    model.eval()

metrics_val, pred_rows_val = evaluate_split(val_records, 'val', model, verbose=True)
metrics_test, pred_rows_test = evaluate_split(test_records, 'test', model, verbose=True)

print('\n=== Robustness checks — test forecast +1 by corridor / season ===')
if pred_rows_test is not None and len(pred_rows_test) and pred_rows_test['parsed'].any():
    parsed_test = pred_rows_test[pred_rows_test['parsed']].copy()
    for group_col in ['corridor_id', 'season', 'source_month', 'target_month']:
        print(f'\nBy {group_col}:')
        for key, g in parsed_test.groupby(group_col):
            mae, rmse, mape = metrics_from_pairs(g['true'], g['pred'])
            print(f'  {str(key):20s} n={len(g):2d}  MAE={mae:.6f}  RMSE={rmse:.6f}  MAPE={mape:.2f}%')
else:
    print('  No parsed test predictions for grouped robustness checks.')


def forecast_plus1_baselines(split_df, split_name):
    train_df = all_meta[all_meta['split'] == 'train'].copy()
    corr_mean = train_df.groupby('corr_id')[PRIMARY_TARGET].mean()
    corr_month_mean = train_df.groupby(['corr_id', 'month'])[PRIMARY_TARGET].mean()
    global_mean = float(train_df[PRIMARY_TARGET].mean())

    rows = []
    for _, row in split_df.iterrows():
        true_val, target_year, target_month, target_meta = get_future_truth(row, all_meta)
        if true_val is None:
            continue
        corr_id = int(row['corr_id'])
        preds = {
            'PERSIST_current_month': float(row[PRIMARY_TARGET]),
            'TRAIN_corr_mean': float(corr_mean.get(corr_id, global_mean)),
            'TRAIN_same_corr_month': float(corr_month_mean.get((corr_id, int(target_month)), corr_mean.get(corr_id, global_mean))),
            'TRAIN_global_mean': global_mean,
        }
        for name, pred in preds.items():
            rows.append({
                'split': split_name,
                'baseline': name,
                'corr_id': corr_id,
                'corridor_id': row['corridor'],
                'source_year': int(row['year']),
                'source_month': int(row['month']),
                'target_year': int(target_year),
                'target_month': int(target_month),
                'true': true_val,
                'pred': pred,
                'abs_error': abs(true_val - pred),
            })
    detail = pd.DataFrame(rows)
    summary = []
    if len(detail):
        for name, g in detail.groupby('baseline'):
            mae, rmse, mape = metrics_from_pairs(g['true'], g['pred'])
            summary.append({'split': split_name, 'baseline': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'n': len(g)})
    return pd.DataFrame(summary), detail


fb_val, fb_detail_val = forecast_plus1_baselines(val_meta, 'val')
fb_test, fb_detail_test = forecast_plus1_baselines(test_meta, 'test')
forecast_baseline_results = pd.concat([fb_val, fb_test], ignore_index=True)
forecast_baseline_details = pd.concat([fb_detail_val, fb_detail_test], ignore_index=True)

print('\n=== Baselines comparables — forecast +1 mois ===')
for split_name, g in forecast_baseline_results.groupby('split'):
    print(f'\n[{split_name}]')
    for _, r in g.sort_values('MAE').iterrows():
        print(f'  {r["baseline"]:<24s} MAE={r["MAE"]:.6f}  RMSE={r["RMSE"]:.6f}  MAPE={r["MAPE"]:.2f}%  n={int(r["n"])}')
    llm_label = f'LLM-{CONFIG["experiment_name"]}'
    if split_name == 'test' and metrics_test:
        print(f'  {llm_label:<24s} MAE={metrics_test["MAE"]:.6f}  RMSE={metrics_test["RMSE"]:.6f}  MAPE={metrics_test["MAPE"]:.2f}%  n={metrics_test["n"]}')
    if split_name == 'val' and metrics_val:
        print(f'  {llm_label:<24s} MAE={metrics_val["MAE"]:.6f}  RMSE={metrics_val["RMSE"]:.6f}  MAPE={metrics_val["MAPE"]:.2f}%  n={metrics_val["n"]}')

print('\n=== Comparaison Phase 3 historique — non directement comparable ===')
print('Note: Phase 3 baselines are same-month traffic-internal baselines; LLM metric here is forecast +1 month.')
baseline_path = pathlib.Path('../data/baseline_results.csv')
if baseline_path.exists():
    base_res = pd.read_csv(baseline_path)
    base_primary = base_res[(base_res['split'] == 'test') & (base_res['target'] == PRIMARY_TARGET)]
    for _, r in base_primary.sort_values('MAE').iterrows():
        print(f'  {r["baseline"]:<4s}: MAE={r["MAE"]:.6f}  RMSE={r["RMSE"]:.6f}  MAPE={r["MAPE"]:.2f}%')
else:
    print('  baseline_results.csv not found — rerun notebook 05 first.')



=== Checkpoint selection — validation forecast MAE ===
  Evaluating checkpoint-30 ...


Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  3.00s/it]


    -> MAE=0.107917  RMSE=0.160429  MAPE=8.2780%
  Evaluating checkpoint-90 ...


Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.88s/it]


    -> MAE=0.112083  RMSE=0.152329  MAPE=8.3050%

Best checkpoint by val forecast MAE: checkpoint-30  (MAE=0.107917)


Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.88s/it]


Selected forecast-best adapter resaved to ../llm/outputs/lora_v10_enriched_yoy_corridors

=== Évaluation numérique [val] — 24 forecast prompts | horizon +1 mois ===
  Airport Road (Route 402) 2024-01 → 2024-02  true=1.2500  pred=1.3100  diff=0.0600
  Airport Road (Route 402) 2024-02 → 2024-03  true=1.2500  pred=1.1800  diff=0.0700
  Airport Road (Route 402) 2024-03 → 2024-04  true=1.2800  pred=1.3100  diff=0.0300
  Airport Road (Route 402) 2024-04 → 2024-05  true=1.3500  pred=1.3100  diff=0.0400
  Airport Road (Route 402) 2024-05 → 2024-06  true=1.3000  pred=1.1800  diff=0.1200
  Airport Road (Route 402) 2024-06 → 2024-07  true=1.2900  pred=1.3100  diff=0.0200
  Patong Hill (Route 4029) 2024-01 → 2024-02  true=1.3100  pred=1.1800  diff=0.1300
  Patong Hill (Route 4029) 2024-02 → 2024-03  true=1.1800  pred=1.1800  diff=0.0000
  Patong Hill (Route 4029) 2024-03 → 2024-04  true=1.0400  pred=1.3100  diff=0.2700
  Patong Hill (Route 4029) 2024-04 → 2024-05  true=1.2700  pred=1.2800  diff=0.

In [18]:
# ── 14. TEST EXPLAINABILITY 1 — ABLATION FIDELITY ─────────────────────────────
# v6 : matching par mots / expressions entières pour éviter le faux positif rain -> constraints.
# On teste ici la dépendance aux variables météo mesurées, pas les risques structurels du corridor.

WEATHER_SPECIFIC_KEYWORDS = [
    'rain', 'rainfall', 'precipitation', 'mm rain',
    'temperature', 'wind', 'humidity', 'storm'
]

ABLATION_CONTROL_INSTRUCTION = """
[ABLATION CONTROL]
Answer using only variables visible in the prompt.
If measured weather variables (rain, rainfall, precipitation, temperature, wind, humidity) are absent, do not mention them.
"""


def compile_phrase_patterns(keywords):
    patterns = {}
    for kw in keywords:
        expr = re.escape(kw.lower()).replace(r'\ ', r'\s+')
        patterns[kw] = re.compile(r'(?<![a-z])' + expr + r'(?![a-z])')
    return patterns


WEATHER_PATTERNS = compile_phrase_patterns(WEATHER_SPECIFIC_KEYWORDS)


def matched_keywords(text, patterns):
    text_lower = text.lower()
    return [kw for kw, pat in patterns.items() if pat.search(text_lower)]


def add_ablation_control(prompt_text):
    return prompt_text.replace('[ANSWER]', ABLATION_CONTROL_INSTRUCTION + '\n[ANSWER]', 1)


explain_records = [r for r in val_records if r['question_type'] == 'explain']
print(f'Ablation Fidelity test on {len(explain_records)} explain prompts (val)\n')

fidelity_results = []
for row in explain_records[:20]:
    prompt_v3 = add_ablation_control(mask_prompt_blocks(row['text'], 'flow_cal_wx'))
    response_v3 = generate_response(prompt_v3, max_new_tokens=256, deterministic=True)

    prompt_v2 = add_ablation_control(mask_prompt_blocks(row['text'], 'flow_cal'))
    response_v2 = generate_response(prompt_v2, max_new_tokens=256, deterministic=True)

    v3_weather_terms = matched_keywords(response_v3, WEATHER_PATTERNS)
    v2_weather_terms = matched_keywords(response_v2, WEATHER_PATTERNS)
    fidelity_ok = len(v2_weather_terms) == 0

    fidelity_results.append({
        'corridor_id': row['corridor_id'],
        'year': int(row['year']),
        'month': int(row['month']),
        'v3_has_weather': bool(v3_weather_terms),
        'v2_has_weather': bool(v2_weather_terms),
        'v3_weather_terms': ', '.join(v3_weather_terms),
        'v2_weather_terms': ', '.join(v2_weather_terms),
        'fidelity_ok': fidelity_ok,
        'response_v3': response_v3,
        'response_v2': response_v2,
    })
    print(f'  {str(row["corridor_id"]):20s}  v3_wx={bool(v3_weather_terms)}  v2_wx={bool(v2_weather_terms)}  ok={fidelity_ok}')

ablation_details_df = pd.DataFrame(fidelity_results)
n_ok = int(ablation_details_df['fidelity_ok'].sum()) if len(ablation_details_df) else 0
n_tot = len(ablation_details_df)
score = n_ok / n_tot * 100 if n_tot > 0 else 0
target = 80

print(f'\nAblation Fidelity score : {n_ok}/{n_tot} = {score:.1f}%')
print(f'Target (proposal)       : ≥{target}%')
print(f'Result                  : {"PASS ✓" if score >= target else "FAIL ✗"}')

if score < target and n_tot:
    bad = next((r for r in fidelity_results if not r['fidelity_ok']), None)
    if bad:
        print('\nExample failed v2 response snippet:')
        print(bad['response_v2'][:500])
        print(f'Detected weather terms: {bad["v2_weather_terms"]}')


Ablation Fidelity test on 24 explain prompts (val)

  Airport Road (Route 402)  v3_wx=False  v2_wx=False  ok=True
  Airport Road (Route 402)  v3_wx=False  v2_wx=False  ok=True
  Airport Road (Route 402)  v3_wx=False  v2_wx=False  ok=True
  Airport Road (Route 402)  v3_wx=False  v2_wx=False  ok=True
  Airport Road (Route 402)  v3_wx=False  v2_wx=False  ok=True
  Airport Road (Route 402)  v3_wx=False  v2_wx=False  ok=True
  Patong Hill (Route 4029)  v3_wx=False  v2_wx=False  ok=True
  Patong Hill (Route 4029)  v3_wx=False  v2_wx=False  ok=True
  Patong Hill (Route 4029)  v3_wx=False  v2_wx=False  ok=True
  Patong Hill (Route 4029)  v3_wx=False  v2_wx=False  ok=True
  Patong Hill (Route 4029)  v3_wx=False  v2_wx=False  ok=True
  Patong Hill (Route 4029)  v3_wx=False  v2_wx=False  ok=True
  Phuket Town → Rawai (Route 4022)  v3_wx=False  v2_wx=False  ok=True
  Phuket Town → Rawai (Route 4022)  v3_wx=False  v2_wx=False  ok=True
  Phuket Town → Rawai (Route 4022)  v3_wx=False  v2_wx=False  ok

In [19]:
# ── 15. TEST EXPLAINABILITY 2 — COUNTERFACTUAL WHAT-IF ────────────────────────
# v6 : pas de few-shot externe ; on teste directement le modèle fine-tuné.

COUNTERFACTUAL_CONTROL_INSTRUCTION = """
[COUNTERFACTUAL CONTROL]
State whether congestion increases, stays roughly stable, or decreases.
If you estimate TT ratios, keep them plausible and consistent with the prompt.
"""


def modify_rainfall_double(prompt_text):
    blocks = parse_blocks(prompt_text)
    if '[CONTEXT]' in blocks:
        blocks['[CONTEXT]'] = re.sub(
            r'([\d.]+) mm rain',
            lambda r: f"{float(r.group(1))*2:.1f} mm rain",
            blocks['[CONTEXT]']
        )
    return reconstruct_prompt(blocks)


def add_counterfactual_control(prompt_text):
    return prompt_text.replace('[ANSWER]', COUNTERFACTUAL_CONTROL_INSTRUCTION + '\n[ANSWER]', 1)


POSITIVE_CONGESTION_WORDS = [
    'increase', 'higher', 'worse', 'slower', 'more congestion', 'heavier',
    'significant impact', 'greater delay', 'longer travel', 'reduced speed',
    'more traffic', 'elevated', 'flood risk', 'delay'
]
NEGATIVE_DECREASE_WORDS = [
    'decrease', 'improve', 'less congestion', 'faster',
    'better condition', 'reduced congestion'
]


def check_counterfactual_direction(response):
    resp_lower = response.lower()
    has_positive = any(kw in resp_lower for kw in POSITIVE_CONGESTION_WORDS)
    has_negative = any(kw in resp_lower for kw in NEGATIVE_DECREASE_WORDS)
    if has_positive and not has_negative:
        return True, 'increase_only'
    elif has_positive and has_negative:
        return True, 'mixed_but_positive'
    elif not has_positive and not has_negative:
        return True, 'neutral'
    else:
        return False, 'decrease_only'


whatif_records = [r for r in val_records if r['question_type'] == 'whatif']
print(f'Counterfactual test on {len(whatif_records[:20])} whatif prompts (val)\n')

counterfactual_results = []
for row in whatif_records[:20]:
    prompt_doubled = add_counterfactual_control(modify_rainfall_double(row['text']))
    response = generate_response(prompt_doubled, max_new_tokens=256, deterministic=True)

    ok, direction = check_counterfactual_direction(response)
    counterfactual_results.append({
        'corridor_id': row['corridor_id'],
        'year': int(row['year']),
        'month': int(row['month']),
        'coherent': ok,
        'direction': direction,
        'response': response,
    })
    print(f'  {str(row["corridor_id"]):20s}  coherent={ok}  direction={direction}')

counterfactual_details_df = pd.DataFrame(counterfactual_results)
n_ok = int(counterfactual_details_df['coherent'].sum()) if len(counterfactual_details_df) else 0
n_tot = len(counterfactual_details_df)
score = n_ok / n_tot * 100 if n_tot > 0 else 0

print(f'\nCounterfactual score : {n_ok}/{n_tot} = {score:.1f}%')
print(f'Target (proposal)    : ≥80%')
print(f'Result               : {"PASS ✓" if score >= 80 else "FAIL ✗"}')


Counterfactual test on 20 whatif prompts (val)

  Airport Road (Route 402)  coherent=True  direction=mixed_but_positive
  Airport Road (Route 402)  coherent=True  direction=mixed_but_positive
  Airport Road (Route 402)  coherent=True  direction=mixed_but_positive
  Airport Road (Route 402)  coherent=True  direction=increase_only
  Airport Road (Route 402)  coherent=True  direction=mixed_but_positive
  Airport Road (Route 402)  coherent=True  direction=mixed_but_positive
  Patong Hill (Route 4029)  coherent=True  direction=mixed_but_positive
  Patong Hill (Route 4029)  coherent=True  direction=mixed_but_positive
  Patong Hill (Route 4029)  coherent=True  direction=mixed_but_positive
  Patong Hill (Route 4029)  coherent=True  direction=mixed_but_positive
  Patong Hill (Route 4029)  coherent=True  direction=mixed_but_positive
  Patong Hill (Route 4029)  coherent=True  direction=mixed_but_positive
  Phuket Town → Rawai (Route 4022)  coherent=True  direction=mixed_but_positive
  Phuket Town

In [20]:
# ── 16. TEST EXPLAINABILITY 3 — GROUNDING CHECK (HALLUCINATION DETECTION) ─────
# v6 : matching strict par mots/expressions entières et comparaison à ce qui est visible dans le prompt masqué.

GROUNDING_CONTROL_INSTRUCTION = """
[GROUNDING CONTROL]
Answer using only the visible traffic profile and corridor description.
If a feature is not shown, do not invent it.
"""

GROUNDING_NEUTRAL_SYSTEM = """ You are an expert traffic analyst for Phuket island, Thailand.
Use only the variables explicitly visible in the prompt.
"""

HALLUCINATION_KEYWORDS_FLOW_ONLY = [
    'passengers', 'flights', 'arrivals', 'hkt',
    'rainfall', 'rain', 'mm rain', 'temperature', 'wind', 'humidity',
    'google trends', 'trends', 'search interest',
    'holiday', 'holidays', 'event', 'events', 'festival',
]

GROUNDING_PATTERNS = compile_phrase_patterns(HALLUCINATION_KEYWORDS_FLOW_ONLY)


def neutralize_system_for_grounding(prompt_text):
    blocks = parse_blocks(prompt_text)
    if '[SYSTEM]' in blocks:
        blocks['[SYSTEM]'] = GROUNDING_NEUTRAL_SYSTEM
    return reconstruct_prompt(blocks)


def add_grounding_control(prompt_text):
    return prompt_text.replace('[ANSWER]', GROUNDING_CONTROL_INSTRUCTION + '\n[ANSWER]', 1)


def check_grounding(response_text, visible_prompt_text):
    resp_lower = response_text.lower()
    prompt_lower = get_question_prompt(visible_prompt_text).lower()
    hallucinated = []
    for kw, pat in GROUNDING_PATTERNS.items():
        if pat.search(resp_lower) and not pat.search(prompt_lower):
            hallucinated.append(kw)
    return len(hallucinated) == 0, hallucinated


print('Grounding test (flow_only variant) on val explain prompts\n')
grounding_results = []

for row in explain_records[:20]:
    prompt_flow_visible = neutralize_system_for_grounding(mask_prompt_blocks(row['text'], 'flow_only'))
    prompt_flow = add_grounding_control(prompt_flow_visible)
    response = generate_response(prompt_flow, max_new_tokens=256, deterministic=True)
    ok, hallucinated = check_grounding(response, prompt_flow_visible)
    grounding_results.append({
        'corridor_id': row['corridor_id'],
        'year': int(row['year']),
        'month': int(row['month']),
        'grounded': ok,
        'hallucinated_vars': ', '.join(hallucinated),
        'response': response,
    })
    status = '✓' if ok else f'✗ hallucinated: {hallucinated[:3]}'
    print(f'  {str(row["corridor_id"]):20s}  {status}')

grounding_details_df = pd.DataFrame(grounding_results)
n_ok = int(grounding_details_df['grounded'].sum()) if len(grounding_details_df) else 0
n_tot = len(grounding_details_df)
score = n_ok / n_tot * 100 if n_tot > 0 else 0

print(f'\nGrounding score : {n_ok}/{n_tot} = {score:.1f}%')
print('(Higher = fewer hallucinated variables when context is removed)')


Grounding test (flow_only variant) on val explain prompts

  Airport Road (Route 402)  ✓
  Airport Road (Route 402)  ✓
  Airport Road (Route 402)  ✓
  Airport Road (Route 402)  ✓
  Airport Road (Route 402)  ✓
  Airport Road (Route 402)  ✗ hallucinated: ['arrivals', 'events']
  Patong Hill (Route 4029)  ✓
  Patong Hill (Route 4029)  ✓
  Patong Hill (Route 4029)  ✓
  Patong Hill (Route 4029)  ✓
  Patong Hill (Route 4029)  ✓
  Patong Hill (Route 4029)  ✓
  Phuket Town → Rawai (Route 4022)  ✓
  Phuket Town → Rawai (Route 4022)  ✓
  Phuket Town → Rawai (Route 4022)  ✓
  Phuket Town → Rawai (Route 4022)  ✗ hallucinated: ['events']
  Phuket Town → Rawai (Route 4022)  ✓
  Phuket Town → Rawai (Route 4022)  ✓
  Bypass Road (Route 4027)  ✓
  Bypass Road (Route 4027)  ✓

Grounding score : 18/20 = 90.0%
(Higher = fewer hallucinated variables when context is removed)


In [21]:
# -- 17. SAVE RESULTS + FINAL SUMMARY ----------------------------------------

experiment_label = f'LLM-{CONFIG["experiment_name"]}'

llm_metrics = []
if metrics_val:
    llm_metrics.append({**metrics_val, 'model': experiment_label})
if metrics_test:
    llm_metrics.append({**metrics_test, 'model': experiment_label})
if llm_metrics:
    pd.DataFrame(llm_metrics).to_csv(CONFIG['results_path'], index=False)
    print(f'LLM metrics saved: {CONFIG["results_path"]}')
if 'pred_rows_test' in globals() and pred_rows_test is not None and len(pred_rows_test):
    pred_rows_test.to_csv(CONFIG['predictions_path'], index=False)
    print(f'LLM per-row forecast predictions saved: {CONFIG["predictions_path"]}')

if 'forecast_baseline_results' in globals() and forecast_baseline_results is not None and len(forecast_baseline_results):
    forecast_baseline_results.to_csv(CONFIG['forecast_baseline_summary_path'], index=False)
    print(f'Forecast +1 baseline summary saved: {CONFIG["forecast_baseline_summary_path"]}')
if 'forecast_baseline_details' in globals() and forecast_baseline_details is not None and len(forecast_baseline_details):
    forecast_baseline_details.to_csv(CONFIG['forecast_baseline_details_path'], index=False)
    print(f'Forecast +1 baseline details saved: {CONFIG["forecast_baseline_details_path"]}')
if 'checkpoint_selection_df' in globals() and checkpoint_selection_df is not None and len(checkpoint_selection_df):
    checkpoint_selection_df.to_csv(CONFIG['checkpoint_selection_path'], index=False)
    print(f'Checkpoint selection saved: {CONFIG["checkpoint_selection_path"]}')
if 'BEST_FORECAST_CHECKPOINT_INFO' in globals() and BEST_FORECAST_CHECKPOINT_INFO:
    pd.DataFrame([BEST_FORECAST_CHECKPOINT_INFO]).to_csv(CONFIG['best_checkpoint_path'], index=False)
    print(f'Best forecast checkpoint saved: {CONFIG["best_checkpoint_path"]}')

expl_summary = {
    'experiment_name': CONFIG['experiment_name'],
    'ablation_fidelity_score': sum(r['fidelity_ok'] for r in fidelity_results) / max(len(fidelity_results), 1) * 100,
    'ablation_fidelity_n': len(fidelity_results),
    'counterfactual_score': sum(r['coherent'] for r in counterfactual_results) / max(len(counterfactual_results), 1) * 100,
    'counterfactual_n': len(counterfactual_results),
    'grounding_score': sum(r['grounded'] for r in grounding_results) / max(len(grounding_results), 1) * 100,
    'grounding_n': len(grounding_results),
}
pd.DataFrame([expl_summary]).to_csv(CONFIG['expl_path'], index=False)
print(f'Explainability scores saved: {CONFIG["expl_path"]}')

if 'ablation_details_df' in globals() and len(ablation_details_df):
    ablation_details_df.to_csv(CONFIG['ablation_details_path'], index=False)
    print(f'Ablation fidelity details saved: {CONFIG["ablation_details_path"]}')
if 'counterfactual_details_df' in globals() and len(counterfactual_details_df):
    counterfactual_details_df.to_csv(CONFIG['counterfactual_details_path'], index=False)
    print(f'Counterfactual details saved: {CONFIG["counterfactual_details_path"]}')
if 'grounding_details_df' in globals() and len(grounding_details_df):
    grounding_details_df.to_csv(CONFIG['grounding_details_path'], index=False)
    print(f'Grounding details saved: {CONFIG["grounding_details_path"]}')

ZERO_SHOT_RESULTS = globals().get('ZERO_SHOT_RESULTS', {})
ZERO_SHOT_ROWS = globals().get('ZERO_SHOT_ROWS', None)
zs_path = pathlib.Path(CONFIG['zs_path'])
if not ZERO_SHOT_RESULTS and zs_path.exists():
    zs_df = pd.read_csv(zs_path)
    ZERO_SHOT_ROWS = zs_df.to_dict('records')
    ZERO_SHOT_RESULTS = {
        r['label']: None if str(r.get('status', '')).upper() != 'OK' else r.to_dict()
        for _, r in zs_df.iterrows()
    }
    print(f'Zero-shot results loaded from disk: {CONFIG["zs_path"]}')

if ZERO_SHOT_ROWS:
    pd.DataFrame(ZERO_SHOT_ROWS).to_csv(CONFIG['zs_path'], index=False)
    print(f'Zero-shot results saved: {CONFIG["zs_path"]}')
elif ZERO_SHOT_RESULTS:
    zs_rows = []
    for lbl, r in ZERO_SHOT_RESULTS.items():
        if r is None:
            zs_rows.append({'label': lbl, 'status': 'SKIP/FAIL'})
        else:
            zs_rows.append(r)
    pd.DataFrame(zs_rows).to_csv(CONFIG['zs_path'], index=False)
    print(f'Zero-shot results saved: {CONFIG["zs_path"]}')
else:
    print('Zero-shot results not in memory and not found on disk - skipped.')

print('\n' + '=' * 70)
print('PHASE 4 - RESULTATS COMPLETS')
print('=' * 70)

print('\n1. MODELE FINE-TUNE')
print(f'   Experiment : {CONFIG["experiment_name"]}')
print(f'   Base       : {CONFIG["model_name"]}')
print(f'   LoRA       : r={CONFIG["lora_r"]}, alpha={CONFIG["lora_alpha"]}')
print(f'   Quant      : {"4-bit QLoRA" if CONFIG["load_in_4bit"] else "8-bit"}')
print(f'   Saved      : {CONFIG["output_dir"]}')
if BEST_FORECAST_CHECKPOINT_INFO:
    print(f'   Best ckpt  : {BEST_FORECAST_CHECKPOINT_INFO.get("checkpoint_name", BEST_FORECAST_CHECKPOINT_INFO.get("checkpoint", "n/a"))}')
    if 'MAE' in BEST_FORECAST_CHECKPOINT_INFO and pd.notna(BEST_FORECAST_CHECKPOINT_INFO['MAE']):
        print(f'   Val MAE ckpt: {BEST_FORECAST_CHECKPOINT_INFO["MAE"]:.6f}')

print('\n2. TABLE FINALE - forecast +1 mois, tt_ratio_Weekday_AM1')
print('-' * 78)
print(f'  {"Modele":<32} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}  Type')
print(f'  {"-"*32} {"-"*10} {"-"*10} {"-"*10}  {"-"*18}')

if 'forecast_baseline_results' in globals() and forecast_baseline_results is not None and len(forecast_baseline_results):
    fb_test = forecast_baseline_results[forecast_baseline_results['split'] == 'test']
    for _, r in fb_test.sort_values('MAE').iterrows():
        print(f'  {r["baseline"]:<32} {r["MAE"]:>10.6f} {r["RMSE"]:>10.6f} {r["MAPE"]:>10.4f}  Forecast baseline')
else:
    print(f'  {"Forecast +1 baselines":<32} {"-":>10} {"-":>10} {"-":>10}  rerun cell 13')

if metrics_test:
    print(f'  {experiment_label + " (fine-tuned)":<32} {metrics_test["MAE"]:>10.6f} {metrics_test["RMSE"]:>10.6f} {metrics_test["MAPE"]:>10.4f}  LLM-fine-tuned *')
print('-' * 78)

print('\n3. ZERO-SHOT LLM - VALIDATION SET')
print('   Note: la cellule 6b evalue les zero-shot sur val (24 prompts), pas sur test; ne pas les melanger aux MAE test ci-dessus.')
print('-' * 78)
print(f'  {"Modele":<32} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}  Type')
paper_models = {'Llama2-7B-chat', 'Llama2-13B-chat', 'Llama2-70B-chat', 'GPT-3.5-turbo', 'GPT-4o'}
for lbl, r in ZERO_SHOT_RESULTS.items():
    tag = 'LLM-zero-shot (papier)' if lbl in paper_models else 'LLM-zero-shot (ajout)'
    if r:
        print(f'  {lbl:<32} {float(r["MAE"]):>10.6f} {float(r["RMSE"]):>10.6f} {float(r["MAPE"]):>10.4f}  {tag}')
    else:
        print(f'  {lbl:<32} {"-":>10} {"-":>10} {"-":>10}  {tag} (skip)')
print('-' * 78)

print('\n4. BASELINES PHASE 3 - NON DIRECTEMENT COMPARABLES')
print('   Ces baselines sont conservees pour reference, mais elles ne remplacent pas les baselines forecast +1 ci-dessus.')
baseline_path = pathlib.Path('../data/baseline_results.csv')
if baseline_path.exists():
    base_res = pd.read_csv(baseline_path)
    base_primary = base_res[(base_res['split'] == 'test') & (base_res['target'] == 'tt_ratio_Weekday_AM1')]
    for _, r in base_primary.sort_values('MAE').iterrows():
        label = 'XGBoost + Optuna' if r['baseline'] == 'XGB' else r['baseline']
        print(f'   {label:<20s} MAE={r["MAE"]:.6f}  RMSE={r["RMSE"]:.6f}  MAPE={r["MAPE"]:.2f}%')
else:
    print('   baseline_results.csv not found - rerun notebook 05 first.')

print('\n5. TESTS EXPLAINABILITY')
print(f'   Ablation Fidelity : {expl_summary["ablation_fidelity_score"]:.1f}%  (target >=80%)')
print(f'   Counterfactual    : {expl_summary["counterfactual_score"]:.1f}%  (target >=80%)')
print(f'   Grounding Check   : {expl_summary["grounding_score"]:.1f}%')

print('\n6. LIMITATIONS DOCUMENTEES')
print(f'   - {len(train_ds)} prompts train dans une branche scratch forecast-only')
print(f'   - Evaluation numerique deterministe sans few-shot externe; verifier {CONFIG["predictions_path"]}')
print('   - Le meilleur checkpoint est choisi sur val forecast MAE, pas seulement sur eval_loss global')
print('   - Les tests explainability restent des proxies via masking, pas 4 modeles d ablation re-entraines')
print("   - Dataset mensuel seulement (96 rows before textualization); plus d historique TomTom ameliorerait la robustesse")
print('   - Si 04_textualize change, rerun 04 puis rerun ce notebook')

print('\n' + '=' * 70)
print('Phase 4 complete.')
print('=' * 70)


LLM metrics saved: ../data/llm_results_v10_enriched_yoy_corridors.csv
LLM per-row forecast predictions saved: ../data/llm_predictions_test_forecast_h1_v10_enriched_yoy_corridors.csv
Forecast +1 baseline summary saved: ../data/forecast_plus1_baselines_v10_enriched_yoy_corridors.csv
Forecast +1 baseline details saved: ../data/forecast_plus1_baseline_details_v10_enriched_yoy_corridors.csv
Checkpoint selection saved: ../data/forecast_checkpoint_scores_v10_enriched_yoy_corridors.csv
Best forecast checkpoint saved: ../data/best_forecast_checkpoint_v10_enriched_yoy_corridors.csv
Explainability scores saved: ../data/explainability_scores_v10_enriched_yoy_corridors.csv
Ablation fidelity details saved: ../data/ablation_fidelity_details_v10_enriched_yoy_corridors.csv
Counterfactual details saved: ../data/counterfactual_details_v10_enriched_yoy_corridors.csv
Grounding details saved: ../data/grounding_details_v10_enriched_yoy_corridors.csv
Zero-shot results saved: ../llm/outputs/zero_shot_results_v

## Conclusions Phase 4

### Ce qui a été implémenté
| Composant | Statut | Détail |
|-----------|--------|--------|
| QLoRA fine-tuning | ✓ | Meta-Llama-3.1-8B-Instruct, r=16, α=32, adaptation 4-bit/petit dataset |
| SFT avec completion-only loss | ✓ | Eq. 3 du papier — loss sur [ANSWER] uniquement |
| Inférence 6 types questions | ✓ | nowcast, forecast, explain, whatif, decision, tourism |
| Few-shot alignment | ✓ | 2 exemples insérés (technique Section 3.4 du papier) |
| Évaluation MAE/RMSE/MAPE | ✓ | Sur forecast +1 mois val/test, parsé par regex + sauvegarde per-row |
| Ablation Fidelity | ✓ | v2 vs v3, score météo keywords |
| Counterfactual | ✓ | Pluie ×2 → vérification direction cohérente |
| Grounding Check | ✓ | Détection hallucinations variables |
| Ablation variants | ✓ | 4 variantes configurables via CONFIG['ablation_variant'] |

### Données TomTom actuelles et extension future
Le pipeline actuel utilise déjà les valeurs TomTom mensuelles réelles 2023-2024 (24 mois × 4 corridors × 24 timesets).

Si un historique plus long arrive (ex. 36 mois ou granularité plus fine) :
1. Rerun `build_master.py` + `03_prepare_dataset` + `04_textualize` → nouveaux prompts
2. Relancer ce notebook avec un nouvel `output_dir`
3. Comparer les résultats : plus d’historique devrait rendre les métriques LLM plus robustes

### Pour la thèse
- Chapitre 4 (Méthodologie) : documenter CONFIG, SFTTrainer setup, few-shot technique
- Chapitre 5 (Résultats) : tableau baselines Phase 3 vs LLM Phase 4, 3 scores explainability
- Limitation principale : 288 prompts train (48 rows × 6 types) (vs ~8.7M papier) → underfitting LLM possible
- MAE numérique LLM à interpréter avec `parse_rate` — si parse_rate < 80%, résultat peu fiable
- Contribution principale : framework démontré sur données réelles TomTom 2023-2024 + explicabilité + 6 types questions chatbot